# Multi-Tenant Caching with Medha

A single Medha backend (Qdrant instance) can serve multiple tenants, applications,
or query languages at the same time. Each **collection name** is a fully isolated
namespace — different vector spaces, different entries, different thresholds.

This notebook covers three common multi-tenant patterns:

1. **Per-application isolation** — SQL app and Cypher app share one Qdrant instance
2. **Per-customer isolation** — SaaS: each customer has their own cache namespace
3. **Shared embedder, isolated collections** — cost-efficient: one embedder object,
   many `Medha` instances

**Requirements:** `pip install medha[fastembed]`

## Cell 1: Imports

In [ ]:
import asyncio
import time

from medha import Medha, Settings, SearchStrategy
from medha.embeddings.fastembed_adapter import FastEmbedAdapter
from medha.types import QueryTemplate

## Cell 2: Pattern 1 — Per-Application Isolation

Two applications (a SQL analytics tool and a Cypher graph explorer) share the same
Qdrant backend but use different collection names. Their caches are completely
independent: a query cached for the SQL app is never returned by the Cypher app.

In [ ]:
# Shared Qdrant settings (same backend, same host)
shared_settings = Settings(qdrant_mode="memory")

# Shared embedder — one model loaded in memory, used by both apps
shared_embedder = FastEmbedAdapter()

# ---- Application 1: SQL analytics ----
medha_sql = Medha(
    collection_name="app_sql_analytics",  # unique namespace
    embedder=shared_embedder,
    settings=shared_settings,
)
await medha_sql.start()

await medha_sql.store_batch([
    {"question": "How many users are there?",   "generated_query": "SELECT COUNT(*) FROM users"},
    {"question": "Show top 10 products",         "generated_query": "SELECT * FROM products LIMIT 10"},
    {"question": "Average order value",          "generated_query": "SELECT AVG(total) FROM orders"},
])
print("SQL app: 3 entries stored")

# ---- Application 2: Cypher graph explorer ----
medha_cypher = Medha(
    collection_name="app_cypher_graph",   # different namespace
    embedder=shared_embedder,
    settings=shared_settings,
)
await medha_cypher.start()

await medha_cypher.store_batch([
    {"question": "How many Person nodes?",       "generated_query": "MATCH (p:Person) RETURN COUNT(p)"},
    {"question": "Find Alice's colleagues",       "generated_query": "MATCH (p:Person {name:'Alice'})-[:WORKS_WITH]-(c) RETURN c.name"},
    {"question": "List all companies",            "generated_query": "MATCH (c:Company) RETURN c.name"},
])
print("Cypher app: 3 entries stored")

In [ ]:
# Isolation test: same question in both apps returns app-specific queries
question = "How many users are there?"

hit_sql    = await medha_sql.search(question)
hit_cypher = await medha_cypher.search(question)

print(f"Question: {question!r}\n")
print(f"SQL app result:")
print(f"  strategy : {hit_sql.strategy}")
print(f"  query    : {hit_sql.generated_query}")
print()
print(f"Cypher app result:")
print(f"  strategy : {hit_cypher.strategy}")
print(f"  query    : {hit_cypher.generated_query if hit_cypher.generated_query else '(no match — correct, different domain)'}")

await medha_sql.close()
await medha_cypher.close()

## Cell 3: Pattern 2 — Per-Customer Isolation (SaaS)

In a SaaS platform, each customer has their own data schema and question vocabulary.
By namespacing collection names with the customer ID, each customer's cache is
completely isolated from others — even if they ask the same questions.

This pattern also allows per-customer settings (different thresholds, different
template files) without any code changes.

In [ ]:
# SaaS tenant registry
TENANTS = {
    "tenant_acme": {
        "pairs": [
            ("How many users?",      "SELECT COUNT(*) FROM acme_users"),
            ("Total revenue",        "SELECT SUM(amount) FROM acme_invoices"),
            ("Active subscriptions", "SELECT COUNT(*) FROM acme_subs WHERE active=1"),
        ],
        "settings": Settings(qdrant_mode="memory", score_threshold_semantic=0.88),
    },
    "tenant_globex": {
        "pairs": [
            ("How many users?",      "SELECT COUNT(*) FROM globex_accounts"),  # same question, different table!
            ("Total revenue",        "SELECT SUM(revenue) FROM globex_deals"),
            ("Pipeline value",       "SELECT SUM(value) FROM globex_pipeline WHERE stage != 'closed'"),
        ],
        "settings": Settings(qdrant_mode="memory", score_threshold_semantic=0.92),
    },
}

shared_embedder = FastEmbedAdapter()

# Startup: initialize one Medha instance per tenant
instances: dict[str, Medha] = {}
for tenant_id, config in TENANTS.items():
    m = Medha(
        collection_name=tenant_id,    # e.g. "tenant_acme"
        embedder=shared_embedder,
        settings=config["settings"],
    )
    await m.start()
    for question, sql in config["pairs"]:
        await m.store(question, sql)
    instances[tenant_id] = m
    print(f"Tenant '{tenant_id}' initialized: {len(config['pairs'])} entries")

print()

In [ ]:
# Per-tenant search: same question, different SQL per tenant
question = "How many users do we have?"

print(f"Question: {question!r}\n")
for tenant_id, medha_instance in instances.items():
    hit = await medha_instance.search(question)
    print(f"  [{tenant_id}]")
    print(f"    strategy : {hit.strategy.value}")
    print(f"    query    : {hit.generated_query}")
    print()

# Cleanup
for m in instances.values():
    await m.close()

## Cell 4: Pattern 3 — Shared Embedder, Isolated Collections

Loading an embedding model into memory is expensive (~200 MB for FastEmbed).
With multiple tenants or applications, you want **one embedder** shared across
all `Medha` instances — not one per tenant.

Since `FastEmbedAdapter` is stateless (no mutable per-request state), it is
safe to share a single instance across multiple `Medha` objects.

In [ ]:
N_TENANTS = 5

# Single embedder shared across all tenants
print("Loading embedder once...")
one_embedder = FastEmbedAdapter()
print(f"  Model: {one_embedder.model_name}, dimension: {one_embedder.dimension}\n")

tenants: list[Medha] = []
for i in range(N_TENANTS):
    m = Medha(
        collection_name=f"tenant_{i:03d}",
        embedder=one_embedder,     # same object, reused
        settings=Settings(qdrant_mode="memory"),
    )
    await m.start()
    await m.store(
        question=f"How many records in tenant {i}?",
        generated_query=f"SELECT COUNT(*) FROM tenant_{i:03d}_records",
    )
    tenants.append(m)

print(f"Initialized {N_TENANTS} tenants using 1 shared embedder instance")
print(f"Memory: 1x model (~{one_embedder.dimension * 4 / 1024:.0f} KB per embedding, model loaded once)\n")

# Verify isolation
for i, m in enumerate(tenants):
    hit = await m.search(f"How many records in tenant {i}?")
    print(f"  tenant_{i:03d}: {hit.strategy.value:18s}  →  {hit.generated_query}")

for m in tenants:
    await m.close()

## Cell 5: Concurrent Multi-Tenant Search

In a real application, multiple tenant requests arrive simultaneously.
`asyncio.gather()` lets you fan out searches across tenants concurrently.

Because Medha is fully async-native, N concurrent tenant searches run in
parallel and complete in roughly the time of a single search.

In [ ]:
N_CONCURRENT = 8
shared_emb = FastEmbedAdapter()

# Setup
tenants = []
for i in range(N_CONCURRENT):
    m = Medha(f"conc_tenant_{i}", embedder=shared_emb, settings=Settings(qdrant_mode="memory"))
    await m.start()
    await m.store(
        question=f"Revenue for customer group {i}",
        generated_query=f"SELECT SUM(amount) FROM orders WHERE group_id={i}",
    )
    tenants.append(m)

# Sequential baseline
t0 = time.perf_counter()
for i, m in enumerate(tenants):
    await m.search(f"Revenue for customer group {i}")
t_sequential = (time.perf_counter() - t0) * 1000

# Reset L1 so both runs are comparable
for m in tenants:
    await m.clear_caches()

# Concurrent: all tenants in parallel
t0 = time.perf_counter()
await asyncio.gather(*[
    m.search(f"Revenue for customer group {i}")
    for i, m in enumerate(tenants)
])
t_concurrent = (time.perf_counter() - t0) * 1000

print(f"{N_CONCURRENT} tenant searches:")
print(f"  Sequential:  {t_sequential:.1f}ms")
print(f"  Concurrent:  {t_concurrent:.1f}ms")
if t_concurrent > 0:
    print(f"  Speedup:     {t_sequential / t_concurrent:.1f}x")

for m in tenants:
    await m.close()

## Cell 6: Per-Tenant Stats

Each `Medha` instance tracks its own stats independently. This lets you monitor
per-tenant hit rates, identify tenants with low cache coverage, and tune
thresholds on a per-tenant basis.

In [ ]:
TENANT_CONFIGS = {
    "tenant_power_user": [
        ("Monthly active users",    "SELECT COUNT(DISTINCT user_id) FROM events WHERE created_at > NOW()-INTERVAL 30 DAY"),
        ("Churn rate this quarter", "SELECT COUNT(*) FROM subscriptions WHERE cancelled_at > DATE_TRUNC('quarter',NOW())"),
        ("NPS score average",       "SELECT AVG(score) FROM nps_responses WHERE created_at > NOW()-INTERVAL 90 DAY"),
    ],
    "tenant_light_user": [
        ("Total orders",            "SELECT COUNT(*) FROM orders"),
    ],
}

shared_emb = FastEmbedAdapter()

print("Simulating mixed tenant workload...\n")
tenant_instances = {}
for tid, pairs in TENANT_CONFIGS.items():
    m = Medha(tid, embedder=shared_emb, settings=Settings(qdrant_mode="memory"))
    await m.start()
    await m.store_batch([{"question": q, "generated_query": sql} for q, sql in pairs])
    tenant_instances[tid] = m

# Simulate workload: power user asks many varied questions, light user repeats
workloads = {
    "tenant_power_user": [
        "Monthly active users",
        "How many users were active last month?",   # semantic
        "Monthly active users",                     # L1
        "Customer churn this quarter",              # semantic of 'churn rate'
        "What is our average NPS?",                 # semantic
        "Something off-topic",                      # miss
    ],
    "tenant_light_user": [
        "Total orders",
        "Total orders",     # L1
        "Total orders",     # L1
    ],
}

for tid, queries in workloads.items():
    m = tenant_instances[tid]
    for q in queries:
        await m.search(q)

# Per-tenant stats report
print(f"  {'Tenant':<25s}  {'Requests':>9s}  {'Hit rate':>9s}  {'L1 hits':>8s}  {'Semantic':>9s}  {'Misses':>7s}")
print("  " + "-" * 80)
for tid, m in tenant_instances.items():
    s = m.stats
    print(f"  {tid:<25s}  {s['total_requests']:>9d}  {s['hit_rate']:>8.1f}%"
          f"  {s['by_strategy']['l1_hits']:>8d}  {s['by_strategy']['semantic_hits']:>9d}"
          f"  {s['by_strategy']['misses']:>7d}")

for m in tenant_instances.values():
    await m.close()

## Cell 7: Tenant Lifecycle — Create, Use, Delete

When a tenant is deactivated or their data must be purged, you can delete their
Qdrant collection. The collection name is the only coupling between Medha and Qdrant.

In [ ]:
from medha.backends.qdrant import QdrantBackend

settings = Settings(qdrant_mode="memory")
shared_emb = FastEmbedAdapter()

# Provision a new tenant
tenant_id = "tenant_trial_001"
m = Medha(tenant_id, embedder=shared_emb, settings=settings)
await m.start()
await m.store("How many items?", "SELECT COUNT(*) FROM items")
print(f"Tenant '{tenant_id}' created and populated")

hit = await m.search("How many items?")
print(f"Search OK: {hit.strategy} — {hit.generated_query}")

# Deactivate tenant: close the Medha instance
await m.close()
print(f"\nTenant '{tenant_id}' deactivated (Medha instance closed)")

# Delete tenant data: drop the Qdrant collection
# In a real deployment with docker/cloud Qdrant, use the Qdrant client directly:
#   from qdrant_client import QdrantClient
#   client = QdrantClient(url="http://localhost:6333")
#   client.delete_collection(tenant_id)
# With in-memory mode, closing the instance already releases all data.
print(f"Tenant '{tenant_id}' data purged (in-memory mode: data released on close)")

## Summary

| Pattern | Collection naming | Use case |
|---|---|---|
| Per-application | `app_sql`, `app_cypher` | Multiple query languages on one backend |
| Per-customer | `tenant_{customer_id}` | SaaS: isolated data per customer |
| Shared embedder | One `FastEmbedAdapter` | Memory-efficient multi-tenant deployments |
| Concurrent search | `asyncio.gather()` | Fan-out per-tenant queries in parallel |
| Per-tenant monitoring | `medha.stats` | Identify low-hit-rate tenants |

The key insight: **Medha collection names are the tenancy boundary**. Everything
else (settings, templates, L1 cache, stats) is scoped per `Medha` instance.